# Grouping, Aggregation, and Pivot Tables

**Course:** Data Science  
**Lesson:** 06  
**Difficulty:** Beginner–Intermediate

## Learning Objectives

By the end of this notebook, students should be able to:

- group data using `groupby()`;
- calculate summary statistics for groups;
- apply multiple aggregation functions;
- group by more than one variable;
- use `transform()` and `filter()`;
- create crosstabs and pivot tables;
- reshape grouped results for reporting;
- interpret grouped outputs correctly.

## Required Libraries

- Pandas
- NumPy
- Matplotlib

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Load the Dataset

This lesson reuses the synthetic employee dataset used in Lessons 04 and 05.

In [ ]:
candidate_paths = [
    Path("../datasets/employee_statistics.csv"),
    Path("Data-Science-Course/datasets/employee_statistics.csv"),
    Path("datasets/employee_statistics.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "employee_statistics.csv was not found. "
        "Place it in Data-Science-Course/datasets/."
    )

df = pd.read_csv(data_path, parse_dates=["Hire_Date"])
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## 2. Basic Grouping with `groupby()`

`groupby()` divides a dataset into groups based on one or more columns.

In [ ]:
department_groups = df.groupby("Department")
department_groups

A `GroupBy` object stores the grouping logic. An aggregation is then applied to produce a result.

In [ ]:
department_groups["Annual_Salary_USD"].mean().round(2)

## 3. Common Aggregation Functions

Common functions include:

- `mean()`
- `median()`
- `sum()`
- `count()`
- `min()`
- `max()`
- `std()`

In [ ]:
department_salary_summary = (
    df.groupby("Department")["Annual_Salary_USD"]
    .agg(["count", "mean", "median", "min", "max", "std"])
    .round(2)
)

department_salary_summary

The summary table provides both group size and salary distribution information.

## 4. Named Aggregations

Named aggregations create readable output column names.

In [ ]:
department_summary = (
    df.groupby("Department")
    .agg(
        Employee_Count=("Employee_ID", "count"),
        Average_Salary=("Annual_Salary_USD", "mean"),
        Median_Salary=("Annual_Salary_USD", "median"),
        Average_Performance=("Performance_Score", "mean"),
        Average_Experience=("Years_Experience", "mean"),
        Total_Projects=("Projects_Completed", "sum"),
    )
    .round(2)
    .sort_values("Average_Salary", ascending=False)
)

department_summary

Named aggregations are useful for reports because the output is easier to interpret.

## 5. Grouping by Multiple Columns

In [ ]:
department_gender_summary = (
    df.groupby(["Department", "Gender"])
    .agg(
        Employee_Count=("Employee_ID", "count"),
        Average_Salary=("Annual_Salary_USD", "mean"),
        Average_Performance=("Performance_Score", "mean"),
    )
    .round(2)
)

department_gender_summary

Grouping by multiple columns creates a hierarchical index.

## 6. Resetting the Index

In [ ]:
department_gender_table = department_gender_summary.reset_index()
department_gender_table.head()

`reset_index()` converts index levels back into regular columns, which is often more convenient for visualization and export.

## 7. Selecting One Group

In [ ]:
available_departments = sorted(df["Department"].dropna().unique())
available_departments

In [ ]:
selected_department = available_departments[0]
department_groups.get_group(selected_department).head()

`get_group()` retrieves the rows belonging to one specific group.

## 8. Group Sizes

In [ ]:
department_sizes = (
    df.groupby("Department")
    .size()
    .sort_values(ascending=False)
    .rename("Employee_Count")
)

department_sizes

`size()` counts all rows in each group. `count()` counts non-missing values in selected columns.

## 9. Applying Custom Aggregation Functions

In [ ]:
def salary_range(series):
    return series.max() - series.min()

custom_salary_summary = (
    df.groupby("Department")["Annual_Salary_USD"]
    .agg(
        Average_Salary="mean",
        Salary_Range=salary_range,
    )
    .round(2)
)

custom_salary_summary

Custom functions are useful when built-in aggregations do not provide the required metric.

## 10. Using `transform()`

`transform()` returns a result aligned with the original rows.

In [ ]:
df["Department_Average_Salary"] = (
    df.groupby("Department")["Annual_Salary_USD"]
    .transform("mean")
)

df["Salary_Difference_From_Department_Average"] = (
    df["Annual_Salary_USD"] - df["Department_Average_Salary"]
)

df[
    [
        "Employee_ID",
        "Department",
        "Annual_Salary_USD",
        "Department_Average_Salary",
        "Salary_Difference_From_Department_Average",
    ]
].head()

`transform()` is appropriate when group statistics need to be added back to the original dataset.

## 11. Department Salary Rank

In [ ]:
df["Department_Salary_Rank"] = (
    df.groupby("Department")["Annual_Salary_USD"]
    .rank(method="dense", ascending=False)
)

df[
    [
        "Employee_ID",
        "Department",
        "Annual_Salary_USD",
        "Department_Salary_Rank",
    ]
].sort_values(
    ["Department", "Department_Salary_Rank"]
).head(15)

Ranking within groups is a common use of grouped operations.

## 12. Filtering Groups

`filter()` keeps or removes entire groups based on a condition.

In [ ]:
departments_with_high_average_salary = (
    df.groupby("Department")
    .filter(lambda group: group["Annual_Salary_USD"].mean() > 65000)
)

departments_with_high_average_salary["Department"].unique()

This keeps all rows from departments whose average salary satisfies the condition.

## 13. Grouped Percentages

In [ ]:
gender_counts = (
    df.groupby(["Department", "Gender"])
    .size()
    .rename("Count")
    .reset_index()
)

gender_counts["Department_Total"] = (
    gender_counts.groupby("Department")["Count"]
    .transform("sum")
)

gender_counts["Percentage"] = (
    100 * gender_counts["Count"] / gender_counts["Department_Total"]
).round(2)

gender_counts

Grouped percentages are useful for comparing composition across departments of different sizes.

## 14. Crosstabs

A crosstab summarizes the frequency relationship between categorical variables.

In [ ]:
department_gender_crosstab = pd.crosstab(
    df["Department"],
    df["Gender"]
)

department_gender_crosstab

In [ ]:
department_gender_percent = pd.crosstab(
    df["Department"],
    df["Gender"],
    normalize="index"
).mul(100).round(2)

department_gender_percent

Normalizing by rows shows the percentage distribution within each department.

## 15. Pivot Tables

Pivot tables summarize numerical data across categories.

In [ ]:
salary_pivot = pd.pivot_table(
    df,
    values="Annual_Salary_USD",
    index="Department",
    columns="Gender",
    aggfunc="mean",
    margins=True
).round(2)

salary_pivot

The pivot table compares average salaries by department and gender and includes overall totals.

## 16. Multiple Values and Aggregations in a Pivot Table

In [ ]:
performance_pivot = pd.pivot_table(
    df,
    values=["Annual_Salary_USD", "Performance_Score"],
    index="Department",
    columns="Education_Level",
    aggfunc={
        "Annual_Salary_USD": "mean",
        "Performance_Score": "mean",
    }
).round(2)

performance_pivot

Complex pivot tables should be used carefully because they can become difficult to read.

## 17. Handling Missing Combinations

In [ ]:
project_pivot = pd.pivot_table(
    df,
    values="Projects_Completed",
    index="Department",
    columns="Education_Level",
    aggfunc="sum",
    fill_value=0
)

project_pivot

`fill_value=0` replaces missing category combinations with zero where appropriate.

## 18. Sorting Grouped Results

In [ ]:
top_departments_by_performance = (
    df.groupby("Department")["Performance_Score"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

top_departments_by_performance

Sorting makes comparisons and reporting more useful.

## 19. Visualizing Grouped Results

In [ ]:
average_salary_by_department = (
    df.groupby("Department")["Annual_Salary_USD"]
    .mean()
    .sort_values(ascending=False)
)

plt.figure(figsize=(9, 5))
average_salary_by_department.plot(kind="bar")

plt.title("Average Salary by Department")
plt.xlabel("Department")
plt.ylabel("Average Salary (USD)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Grouped summaries are often easier to interpret when visualized.

## 20. Pivot Table Visualization

In [ ]:
pivot_for_plot = pd.pivot_table(
    df,
    values="Performance_Score",
    index="Department",
    columns="Gender",
    aggfunc="mean"
)

pivot_for_plot.plot(kind="bar", figsize=(10, 5))

plt.title("Average Performance Score by Department and Gender")
plt.xlabel("Department")
plt.ylabel("Average Performance Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 21. Practical Reporting Example

In [ ]:
management_report = (
    df.groupby("Department")
    .agg(
        Employees=("Employee_ID", "count"),
        Avg_Salary=("Annual_Salary_USD", "mean"),
        Avg_Performance=("Performance_Score", "mean"),
        Avg_Satisfaction=("Job_Satisfaction", "mean"),
        Total_Projects=("Projects_Completed", "sum"),
        Avg_Absence_Days=("Absence_Days", "mean"),
    )
    .round(2)
)

management_report["Salary_per_Project"] = (
    management_report["Avg_Salary"]
    / management_report["Total_Projects"]
).round(2)

management_report.sort_values(
    "Avg_Performance",
    ascending=False
)

A useful report combines several relevant measures but should avoid including unnecessary statistics.

## 22. Common Mistakes

1. Using `groupby()` without applying an aggregation.
2. Confusing `size()` with `count()`.
3. Ignoring missing values during aggregation.
4. Comparing averages without considering group size.
5. Creating pivot tables that are too complex to interpret.
6. Forgetting to reset a hierarchical index when needed.
7. Using totals where averages or percentages are more appropriate.
8. Interpreting grouped patterns without checking the underlying observations.

## 23. Exercises

1. Calculate average age by department.
2. Find the median salary by education level.
3. Count employees by city and gender.
4. Calculate average performance and satisfaction by department.
5. Rank employees by performance within each department.
6. Add each employee's department-average absence days using `transform()`.
7. Keep only departments with at least 50 employees.
8. Create a crosstab of education level and gender.
9. Create a pivot table of average salary by department and education level.
10. Build a summary report with employee count, average salary, and total projects by department.

In [ ]:
# Write your exercise solutions here.

## 24. Summary

In this notebook, you learned how to:

- group data using one or more variables;
- calculate simple and multiple aggregations;
- use named aggregations;
- add group-level values with `transform()`;
- filter entire groups;
- calculate grouped percentages;
- create crosstabs and pivot tables;
- visualize and report grouped results.

## Next Lesson

The next lesson is **Statistical Hypothesis Testing**.